In [2]:
"""
PIPELINE STAGE: Topographic Data Mining & Numeric Feature Engineering
PERIOD: Static Geographic Covariate
LIBRARIES: os, pandas, python-docx (docx), re

1. RESEARCH OBJECTIVE:
   Automate the extraction of altitude (elevation) data for 81 Turkish provinces from 
   unstructured Word document tables. Altitude serves as a critical time-invariant 
   environmental covariate in spatial energy modeling, directly influencing ambient 
   temperature and residential heating/cooling demands.

2. FILE ARCHITECTURE:
   - Input (Unstructured Doc): os.path.join("..", "..", "raw_data", "altitude", "altitude.docx")
   - Output (Relational Covariate): os.path.join("..", "..", "processed_data", "steps", "14_altitude.xlsx")
   - System: Dynamically generates the output directory structure if it does not exist.

3. METHODOLOGICAL ARCHITECTURE & TABLE MAPPING:
   - Implicit Relational Mapping: Leverages the sequential order of tables within the document 
     by enumerating them (Index 1 to 81) and matching the table index directly to the 
     official Turkish Plate Code dictionary.
   - Textual Targeting: Isolates the 4th row (Index 3) and 1st cell (Index 0) of each matched 
     table, scanning for the target keyword ('rakim').

4. DATA REFINEMENT & REGEX SANITIZATION:
   - String-to-Numeric Transformation: Implements a strict Regex cleaning layer (re.sub(r'[^\d]', '')) 
     to aggressively strip all non-numeric artifacts (dots, colons, text, spaces) from the raw 
     string (e.g., transforming " : 1.050 " directly into "1050").

5. DATA INTEGRITY & TYPE ENFORCEMENT:
   - Error Handling: Applies 'pd.to_numeric(..., errors='coerce')' to identify unparseable data 
     points, immediately followed by 'dropna' to purge missing values and preserve dataset integrity.
   - Strict Integer Casting: Enforces a final Integer (int) type conversion for the 'Altitude' 
     column, ensuring seamless integration into downstream correlation matrices and regression models.
"""
import docx
import os
import pandas as pd
import re

# Dosya yolları
file_path = os.path.join("..", "..", "raw_data", "altitude", "altitude.docx")
output_excel = os.path.join("..", "..", "processed_data", "steps","14_altitude.xlsx")

# Resmi Plaka ve İl Eşleşme Sözlüğü
plate_map = {
    1: "Adana", 2: "Adıyaman", 3: "Afyonkarahisar", 4: "Ağrı", 5: "Amasya", 6: "Ankara", 7: "Antalya", 8: "Artvin", 9: "Aydın",
    10: "Balıkesir", 11: "Bilecik", 12: "Bingöl", 13: "Bitlis", 14: "Bolu", 15: "Burdur", 16: "Bursa", 17: "Çanakkale", 18: "Çankırı", 19: "Çorum",
    20: "Denizli", 21: "Diyarbakır", 22: "Edirne", 23: "Elazığ", 24: "Erzincan", 25: "Erzurum", 26: "Eskişehir", 27: "Gaziantep", 28: "Giresun", 29: "Gümüşhane",
    30: "Hakkari", 31: "Hatay", 32: "Isparta", 33: "Mersin", 34: "İstanbul", 35: "İzmir", 36: "Kars", 37: "Kastamonu", 38: "Kayseri", 39: "Kırklareli",
    40: "Kırşehir", 41: "Kocaeli", 42: "Konya", 43: "Kütahya", 44: "Malatya", 45: "Manisa", 46: "Kahramanmaraş", 47: "Mardin", 48: "Muğla", 49: "Muş",
    50: "Nevşehir", 51: "Niğde", 52: "Ordu", 53: "Rize", 54: "Sakarya", 55: "Samsun", 56: "Siirt", 57: "Sinop", 58: "Sivas", 59: "Tekirdağ",
    60: "Tokat", 61: "Trabzon", 62: "Tunceli", 63: "Şanlıurfa", 64: "Uşak", 65: "Van", 66: "Yozgat", 67: "Zonguldak", 68: "Aksaray", 69: "Bayburt",
    70: "Karaman", 71: "Kırıkkale", 72: "Batman", 73: "Şırnak", 74: "Bartın", 75: "Ardahan", 76: "Iğdır", 77: "Yalova", 78: "Karabük", 79: "Kilis",
    80: "Osmaniye", 81: "Düzce"
}

def export_to_excel_numeric(path):
    if not os.path.exists(path):
        print(f"Hata: {path} bulunamadı.")
        return

    doc = docx.Document(path)
    data_list = []
    
    for i, table in enumerate(doc.tables, 1):
        if i in plate_map:
            if len(table.rows) >= 4:
                row = table.rows[3]
                if len(row.cells) > 0:
                    raw_text = row.cells[0].text.strip().lower()
                    
                    if "rakim" in raw_text:
                        parts = raw_text.split("rakim")
                        
                        # 1. Ham metni al ve içindeki rakam dışı her şeyi (nokta dahil) temizle
                        # Örn: " : 1.050 " -> "1050"
                        altitude_raw = parts[1].replace(":", "").strip()
                        altitude_clean = re.sub(r'[^\d]', '', altitude_raw)
                        
                        data_list.append({
                            "Plate": i,
                            "Province": plate_map[i],
                            "Altitude": altitude_clean
                        })

    # Veriyi DataFrame'e çevir
    df = pd.DataFrame(data_list)

    # 2. Sayısal Dönüşüm
    # Altitude sütununu tam sayıya çeviriyoruz
    df['Altitude'] = pd.to_numeric(df['Altitude'], errors='coerce')
    
    # Eğer boş değer oluştuysa (temizlenemeyen hatalı veri), onları 0 veya ortalama ile doldurmak yerine atabiliriz
    df = df.dropna(subset=['Altitude'])
    df['Altitude'] = df['Altitude'].astype(int)

    # Excel'e kaydet
    if not df.empty:
        os.makedirs(os.path.dirname(output_excel), exist_ok=True)
        df.to_excel(output_excel, index=False)
        print("\n" + "="*50)
        print(f"BAŞARILI: {len(df)} ilin rakım verisi SAYISAL olarak kaydedildi.")
        print("-" * 50)
        print(df.head(10).to_string(index=False))
        print("="*50)
    else:
        print("Uygun veri bulunamadı.")

if __name__ == "__main__":
    export_to_excel_numeric(file_path)

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ismailgulsoy\AppData\Local\Temp\ipykernel_5880\196424826.py:1: SyntaxWarning: invalid escape sequence '\d'
  """



BAŞARILI: 81 ilin rakım verisi SAYISAL olarak kaydedildi.
--------------------------------------------------
 Plate       Province  Altitude
     1          Adana        23
     2       Adıyaman       669
     3 Afyonkarahisar      1021
     4           Ağrı      1640
     5         Amasya       392
     6         Ankara       850
     7        Antalya        39
     8         Artvin       500
     9          Aydın        64
    10      Balıkesir       120
